In [1]:
# Cell 1 — Load libraries and locate the repository.
#
# This notebook will construct the complete canonical
# one-minute dataset from all validated raw segments.
#
# We locate the project dynamically rather than hardcoding
# C:\Users\... so the repository can be moved later.

from dataclasses import dataclass
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


def find_project_root(start_path):
    """Search upward for the repository root."""

    current_path = Path(start_path).resolve()

    for candidate in [
        current_path,
        *current_path.parents,
    ]:
        if (
            (candidate / ".git").exists()
            and (candidate / "raw_data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate a repository containing "
        "both .git and raw_data."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

RAW_DATA_DIR = PROJECT_ROOT / "raw_data"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
REPORT_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"


print("Python version: ", sys.version.split()[0])
print("Pandas version: ", pd.__version__)
print("PyArrow version:", pa.__version__)

print()
print("Project root:   ", PROJECT_ROOT)
print("Raw data:       ", RAW_DATA_DIR)
print("Interim data:   ", INTERIM_DATA_DIR)
print("Report tables:  ", REPORT_TABLES_DIR)

print()
print(
    "Environment check:",
    "PASS"
    if RAW_DATA_DIR.exists()
    else "FAIL",
)

Python version:  3.13.16
Pandas version:  3.0.6
PyArrow version: 25.0.1

Project root:    C:\Users\Ijzd4\trend models\EURUSD 15
Raw data:        C:\Users\Ijzd4\trend models\EURUSD 15\raw_data
Interim data:    C:\Users\Ijzd4\trend models\EURUSD 15\data\interim
Report tables:   C:\Users\Ijzd4\trend models\EURUSD 15\reports\tables

Environment check: PASS


In [2]:
# Cell 2 — Define centralized instrument configuration.
#
# The processing functions will not hardcode EUR/USD, pip size,
# market calendar, bar length, or forecast horizon.
#
# For another asset, we will create another configuration object
# while reusing the same processing code.

@dataclass(frozen=True)
class InstrumentConfig:
    symbol: str
    asset_class: str
    source: str

    # Smallest normal price increment supplied by the feed.
    price_increment: float

    # Unit used when reporting spreads and price movements.
    # For FX this is one pip.
    reporting_unit: float
    reporting_unit_name: str

    bar_minutes: int
    forecast_horizon_minutes: int

    timezone: str
    market_calendar: str


instrument = InstrumentConfig(
    symbol="EURUSD",
    asset_class="fx",
    source="dukascopy",

    price_increment=0.00001,
    reporting_unit=0.0001,
    reporting_unit_name="pip",

    bar_minutes=1,
    forecast_horizon_minutes=10,

    timezone="UTC",
    market_calendar="fx_24_5",
)


# Confirm that the reporting unit can be represented as an
# exact whole number of minimum price increments.
increments_per_reporting_unit = (
    instrument.reporting_unit
    / instrument.price_increment
)

configuration_ok = (
    instrument.forecast_horizon_minutes == 10
    and instrument.bar_minutes > 0
    and instrument.price_increment > 0
    and instrument.reporting_unit > 0
    and np.isclose(
        increments_per_reporting_unit,
        round(increments_per_reporting_unit),
    )
)


print("Instrument:          ", instrument.symbol)
print("Asset class:         ", instrument.asset_class)
print("Data source:         ", instrument.source)
print("Price increment:     ", instrument.price_increment)
print(
    "Reporting unit:      ",
    instrument.reporting_unit,
    instrument.reporting_unit_name,
)
print("Bar length:          ", instrument.bar_minutes, "minute")
print(
    "Forecast horizon:    ",
    instrument.forecast_horizon_minutes,
    "minutes",
)
print("Market calendar:     ", instrument.market_calendar)

print()
print(
    "Configuration check:",
    "PASS" if configuration_ok else "FAIL",
)

Instrument:           EURUSD
Asset class:          fx
Data source:          dukascopy
Price increment:      1e-05
Reporting unit:       0.0001 pip
Bar length:           1 minute
Forecast horizon:     10 minutes
Market calendar:      fx_24_5

Configuration check: PASS


In [3]:
# Cell 3 — Inventory all EUR/USD raw CSV files.
#
# This cell reads only file metadata and the first line.
# It does not scan the large files or change anything.

EXPECTED_DUKASCOPY_COLUMNS = (
    "timestamp",
    "askPrice",
    "bidPrice",
    "askVolume",
    "bidVolume",
)

EXPECTED_DUKASCOPY_HEADER = ",".join(
    EXPECTED_DUKASCOPY_COLUMNS
)


inventory_records = []

for csv_file in sorted(
    RAW_DATA_DIR.rglob(
        f"{instrument.symbol.lower()}*.csv"
    )
):
    file_size_bytes = csv_file.stat().st_size

    if file_size_bytes == 0:
        header = None
        file_status = "empty"
    else:
        with csv_file.open(
            mode="r",
            encoding="utf-8",
            newline="",
        ) as file_handle:
            header = file_handle.readline().strip()

        file_status = (
            "valid_header"
            if header == EXPECTED_DUKASCOPY_HEADER
            else "unexpected_header"
        )

    inventory_records.append(
        {
            "relative_path": str(
                csv_file.relative_to(PROJECT_ROOT)
            ),
            "filename": csv_file.name,
            "size_gb": (
                file_size_bytes / (1024 ** 3)
            ),
            "size_bytes": file_size_bytes,
            "header": header,
            "status": file_status,
        }
    )


raw_file_inventory = pd.DataFrame(
    inventory_records
)

print(
    "EUR/USD CSV files found:",
    len(raw_file_inventory),
)

display(
    raw_file_inventory[
        [
            "relative_path",
            "size_gb",
            "status",
        ]
    ]
)

EUR/USD CSV files found: 18


,relative_path,size_gb,status
0,raw_data\eurusd-test-bi5-2016-04-05.csv,0.000000,empty
1,raw_data\eurusd-tick-2010-01-01-2026-10-01.csv,4.388993,valid_header
2,raw_data\eurusd-tick-2016-04-05-2017-01-01.csv,0.000565,valid_header
3,raw_data\eurusd-tick-2016-04-05-2026-10-01.csv...,0.000000,empty
4,raw_data\eurusd-tick-2026-09-01-2026-09-02.csv,0.001622,valid_header
5,raw_data\repair_test\eurusd-tick-2016-04-05-20...,0.003367,valid_header
6,raw_data\repair_test\eurusd-tick-2016-04-06-20...,0.016611,valid_header
7,raw_data\repair_test\eurusd-tick-2016-04-13-20...,0.069317,valid_header
8,raw_data\repair_test\eurusd-tick-2016-05-13-20...,0.060629,valid_header
9,raw_data\repair_test\eurusd-tick-2016-06-13-20...,1.143133,valid_header


In [4]:
# Cell 4 — Read actual timestamp boundaries from every valid CSV.
#
# The first row is read normally.
# The final row is found by seeking backward from the end, so
# even the multi-gigabyte files are not loaded into memory.

def read_first_data_line(file_path):
    """Return the first data row after the CSV header."""

    with file_path.open(
        mode="r",
        encoding="utf-8",
        newline="",
    ) as file_handle:
        next(file_handle)  # Skip the header.
        return file_handle.readline().strip()


def read_last_data_line(file_path):
    """Return the final non-empty row using binary file seeking."""

    with file_path.open(mode="rb") as file_handle:
        file_handle.seek(0, 2)
        position = file_handle.tell() - 1

        # Skip trailing newline characters.
        while position >= 0:
            file_handle.seek(position)
            character = file_handle.read(1)

            if character not in (b"\n", b"\r"):
                break

            position -= 1

        line_end = position + 1

        # Search backward for the preceding newline.
        while position >= 0:
            file_handle.seek(position)

            if file_handle.read(1) == b"\n":
                break

            position -= 1

        line_start = position + 1

        file_handle.seek(line_start)

        return file_handle.read(
            line_end - line_start
        ).decode("utf-8").strip()


def timestamp_from_csv_line(csv_line):
    """Extract the millisecond Unix timestamp from a tick row."""

    return int(csv_line.split(",", maxsplit=1)[0])


boundary_records = []

valid_inventory = raw_file_inventory.loc[
    raw_file_inventory["status"] == "valid_header"
]

for relative_path in valid_inventory["relative_path"]:
    csv_file = PROJECT_ROOT / relative_path

    first_line = read_first_data_line(csv_file)
    last_line = read_last_data_line(csv_file)

    first_timestamp = timestamp_from_csv_line(
        first_line
    )
    last_timestamp = timestamp_from_csv_line(
        last_line
    )

    boundary_records.append(
        {
            "relative_path": relative_path,
            "filename": csv_file.name,
            "size_gb": (
                csv_file.stat().st_size
                / (1024 ** 3)
            ),
            "first_timestamp": first_timestamp,
            "last_timestamp": last_timestamp,
            "first_tick": pd.to_datetime(
                first_timestamp,
                unit="ms",
                utc=True,
            ),
            "last_tick": pd.to_datetime(
                last_timestamp,
                unit="ms",
                utc=True,
            ),
        }
    )


file_boundaries = (
    pd.DataFrame(boundary_records)
    .sort_values(
        [
            "first_timestamp",
            "last_timestamp",
        ]
    )
    .reset_index(drop=True)
)


with pd.option_context(
    "display.max_colwidth",
    None,
):
    display(
        file_boundaries[
            [
                "relative_path",
                "size_gb",
                "first_tick",
                "last_tick",
            ]
        ]
    )

,relative_path,size_gb,first_tick,last_tick
0,raw_data\eurusd-tick-2010-01-01-2026-10-01.csv,4.388993,2010-01-01 00:00:03.964000+00:00,2016-04-05 05:59:59.593000+00:00
1,raw_data\eurusd-tick-2016-04-05-2017-01-01.csv,0.000565,2016-04-05 00:00:01.589000+00:00,2016-04-05 05:59:59.593000+00:00
2,raw_data\repair_test\eurusd-tick-2016-04-05-2016-04-06.csv,0.003367,2016-04-05 00:00:01.589000+00:00,2016-04-05 23:59:58.700000+00:00
3,raw_data\repair_test\eurusd-tick-2016-04-06-2016-04-13.csv,0.016611,2016-04-06 00:00:00.202000+00:00,2016-04-12 23:59:59.859000+00:00
4,raw_data\repair_test\eurusd-tick-2016-04-13-2016-05-13.csv,0.069317,2016-04-13 00:00:00.437000+00:00,2016-05-12 23:59:57.359000+00:00
5,raw_data\repair_test\eurusd-tick-2016-05-13-2016-06-13.csv,0.060629,2016-05-13 00:00:00.642000+00:00,2016-06-12 23:59:59.829000+00:00
6,raw_data\repair_test\eurusd-tick-2016-06-13-2017-01-01.csv,1.143133,2016-06-13 00:00:00.369000+00:00,2016-12-30 21:59:56.230000+00:00
7,raw_data\repair_test\eurusd-tick-2017-01-01-2018-01-01.csv,0.764868,2017-01-01 22:00:20.786000+00:00,2017-12-29 21:59:42.990000+00:00
8,raw_data\repair_test\eurusd-tick-2018-01-01-2019-01-01.csv,0.925600,2018-01-01 22:00:08.661000+00:00,2018-12-31 21:59:59.716000+00:00
9,raw_data\repair_test\eurusd-tick-2019-01-01-2020-01-01.csv,1.040274,2019-01-01 22:02:37.254000+00:00,2019-12-31 21:59:57.829000+00:00


In [5]:
# Cell 5 — Classify canonical and redundant source files.
#
# Selection rule:
#   1. Use the valid file with the earliest observed tick as
#      the historical baseline.
#   2. Use every validated continuation segment stored in
#      raw_data/repair_test.
#   3. Exclude other files whose coverage is already contained
#      inside the canonical chain.
#
# This selection concerns this dataset's file organization.
# The tick-processing functions themselves remain asset-agnostic.

baseline_index = (
    file_boundaries["first_timestamp"].idxmin()
)

baseline_file_record = file_boundaries.loc[
    [baseline_index]
].copy()


is_continuation_segment = (
    file_boundaries["relative_path"]
    .map(
        lambda value: (
            "repair_test"
            in Path(value).parts
        )
    )
)

continuation_file_records = (
    file_boundaries.loc[
        is_continuation_segment
    ]
    .copy()
)


canonical_file_table = (
    pd.concat(
        [
            baseline_file_record,
            continuation_file_records,
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset="relative_path")
    .sort_values(
        [
            "first_timestamp",
            "last_timestamp",
        ]
    )
    .reset_index(drop=True)
)


redundant_file_table = (
    file_boundaries.loc[
        ~file_boundaries["relative_path"].isin(
            canonical_file_table["relative_path"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# Confirm that every excluded file is fully contained inside at
# least one canonical file.
def coverage_is_contained(
    excluded_row,
    canonical_table,
):
    return any(
        (
            canonical_row.first_timestamp
            <= excluded_row.first_timestamp
        )
        and (
            canonical_row.last_timestamp
            >= excluded_row.last_timestamp
        )
        for canonical_row
        in canonical_table.itertuples()
    )


redundant_coverage_ok = all(
    coverage_is_contained(
        excluded_row,
        canonical_file_table,
    )
    for excluded_row
    in redundant_file_table.itertuples()
)


# Build a boundary table between consecutive canonical files.
boundary_checks = []

for position in range(
    1,
    len(canonical_file_table),
):
    previous_row = canonical_file_table.iloc[
        position - 1
    ]
    current_row = canonical_file_table.iloc[
        position
    ]

    boundary_seconds = (
        current_row["first_timestamp"]
        - previous_row["last_timestamp"]
    ) / 1000

    if boundary_seconds < 0:
        relationship = "overlap"
    elif boundary_seconds <= 60:
        relationship = "continuous"
    else:
        relationship = "market closure or gap"

    boundary_checks.append(
        {
            "previous_file": previous_row["filename"],
            "next_file": current_row["filename"],
            "boundary_hours": (
                boundary_seconds / 3600
            ),
            "relationship": relationship,
        }
    )


canonical_boundary_table = pd.DataFrame(
    boundary_checks
)

canonical_files = [
    PROJECT_ROOT / relative_path
    for relative_path
    in canonical_file_table["relative_path"]
]


selection_ok = (
    len(canonical_files) > 1
    and redundant_coverage_ok
    and canonical_file_table[
        "first_timestamp"
    ].is_monotonic_increasing
)


print("Canonical files:", len(canonical_file_table))
print("Redundant files:", len(redundant_file_table))
print(
    "Redundant coverage contained:",
    redundant_coverage_ok,
)

print()
print("Excluded redundant files:")

display(
    redundant_file_table[
        [
            "relative_path",
            "first_tick",
            "last_tick",
        ]
    ]
)

print()
print("Canonical file boundaries:")

display(canonical_boundary_table)

print()
print(
    "Canonical selection:",
    "PASS" if selection_ok else "FAIL",
)

Canonical files: 14
Redundant files: 2
Redundant coverage contained: True

Excluded redundant files:


,relative_path,first_tick,last_tick
0,raw_data\eurusd-tick-2016-04-05-2017-01-01.csv,2016-04-05 00:00:01.589000+00:00,2016-04-05 05:59:59.593000+00:00
1,raw_data\eurusd-tick-2026-09-01-2026-09-02.csv,2026-09-01 00:00:00.118000+00:00,2026-09-01 23:59:59.083000+00:00



Canonical file boundaries:


,previous_file,next_file,boundary_hours,relationship
0,eurusd-tick-2010-01-01-2026-10-01.csv,eurusd-tick-2016-04-05-2016-04-06.csv,-5.999446,overlap
1,eurusd-tick-2016-04-05-2016-04-06.csv,eurusd-tick-2016-04-06-2016-04-13.csv,0.000417,continuous
2,eurusd-tick-2016-04-06-2016-04-13.csv,eurusd-tick-2016-04-13-2016-05-13.csv,0.000161,continuous
3,eurusd-tick-2016-04-13-2016-05-13.csv,eurusd-tick-2016-05-13-2016-06-13.csv,0.000912,continuous
4,eurusd-tick-2016-05-13-2016-06-13.csv,eurusd-tick-2016-06-13-2017-01-01.csv,0.000150,continuous
5,eurusd-tick-2016-06-13-2017-01-01.csv,eurusd-tick-2017-01-01-2018-01-01.csv,48.006821,market closure or gap
6,eurusd-tick-2017-01-01-2018-01-01.csv,eurusd-tick-2018-01-01-2019-01-01.csv,72.007131,market closure or gap
7,eurusd-tick-2018-01-01-2019-01-01.csv,eurusd-tick-2019-01-01-2020-01-01.csv,24.043761,market closure or gap
8,eurusd-tick-2019-01-01-2020-01-01.csv,eurusd-tick-2020-01-01-2021-01-01.csv,24.020831,market closure or gap
9,eurusd-tick-2020-01-01-2021-01-01.csv,eurusd-tick-2021-01-01-2022-01-01.csv,72.000256,market closure or gap



Canonical selection: PASS


In [6]:
# Cell 6 — Verify the April 2016 overlap row-for-row.
#
# We use the small redundant file as a compact reference.
# The comparison reads only:
#   - the final overlap rows from the original file,
#   - the small reference file,
#   - the same number of rows from the continuation file.
#
# The 4.4 GB original file is not scanned from the beginning.

from itertools import islice


baseline_row = canonical_file_table.iloc[0]
first_continuation_row = canonical_file_table.iloc[1]


# Find the redundant file whose boundaries exactly match the
# overlap between the baseline and first continuation file.
overlap_reference_candidates = (
    redundant_file_table.loc[
        (
            redundant_file_table["first_timestamp"]
            == first_continuation_row["first_timestamp"]
        )
        & (
            redundant_file_table["last_timestamp"]
            == baseline_row["last_timestamp"]
        )
    ]
)


if len(overlap_reference_candidates) != 1:
    raise ValueError(
        "Expected exactly one compact overlap reference file."
    )


overlap_reference_row = (
    overlap_reference_candidates.iloc[0]
)

baseline_file = (
    PROJECT_ROOT
    / baseline_row["relative_path"]
)

continuation_file = (
    PROJECT_ROOT
    / first_continuation_row["relative_path"]
)

overlap_reference_file = (
    PROJECT_ROOT
    / overlap_reference_row["relative_path"]
)


def read_all_data_lines(file_path):
    """Read data lines from a small CSV, excluding its header."""

    with file_path.open(
        mode="r",
        encoding="utf-8",
        newline="",
    ) as file_handle:
        next(file_handle)

        return [
            line.rstrip("\r\n")
            for line in file_handle
            if line.strip()
        ]


def read_first_n_data_lines(
    file_path,
    row_count,
):
    """Read the first N data rows after the header."""

    with file_path.open(
        mode="r",
        encoding="utf-8",
        newline="",
    ) as file_handle:
        next(file_handle)

        return [
            line.rstrip("\r\n")
            for line in islice(
                file_handle,
                row_count,
            )
        ]


def read_last_n_data_lines(
    file_path,
    row_count,
    block_size=1024 * 1024,
):
    """Read the final N rows without scanning the whole file."""

    with file_path.open(mode="rb") as file_handle:
        file_handle.seek(0, 2)
        remaining_bytes = file_handle.tell()

        collected = b""

        while (
            remaining_bytes > 0
            and len(collected.splitlines())
            <= row_count
        ):
            bytes_to_read = min(
                block_size,
                remaining_bytes,
            )

            remaining_bytes -= bytes_to_read
            file_handle.seek(remaining_bytes)

            collected = (
                file_handle.read(bytes_to_read)
                + collected
            )

    final_lines = [
        line.decode("utf-8")
        for line in collected.splitlines()
        if line.strip()
    ]

    return final_lines[-row_count:]


reference_lines = read_all_data_lines(
    overlap_reference_file
)

overlap_row_count = len(reference_lines)

baseline_tail_lines = read_last_n_data_lines(
    baseline_file,
    overlap_row_count,
)

continuation_prefix_lines = (
    read_first_n_data_lines(
        continuation_file,
        overlap_row_count,
    )
)


baseline_match = (
    baseline_tail_lines == reference_lines
)

continuation_match = (
    continuation_prefix_lines
    == reference_lines
)


print("Overlap rows:                 ", f"{overlap_row_count:,}")
print("Original tail matches:        ", baseline_match)
print("Continuation prefix matches:  ", continuation_match)

print()
print(
    "Overlap verification:",
    (
        "PASS"
        if baseline_match and continuation_match
        else "FAIL"
    ),
)

Overlap rows:                  16,105
Original tail matches:         True
Continuation prefix matches:   True

Overlap verification: PASS


In [7]:
# Cell 7 — Define generic tick preparation.
#
# Prices are converted into integer minimum-price increments.
# This avoids false tick directions caused by binary floating
# point differences.
#
# For EUR/USD:
#   one price tick = 0.00001
#   one reporting unit = one pip = 0.0001
#
# The same function can later process another asset by receiving
# a different InstrumentConfig object.

TICK_DTYPES = {
    "timestamp": "int64",
    "askPrice": "float64",
    "bidPrice": "float64",
    "askVolume": "float64",
    "bidVolume": "float64",
}

RAW_TICK_COLUMNS = list(
    TICK_DTYPES.keys()
)


def prepare_tick_rows(
    raw_ticks,
    instrument_config,
    previous_mid_half_ticks=None,
):
    """Prepare raw quote ticks for time-bar aggregation."""

    prepared = raw_ticks.copy()

    prepared["event_time"] = pd.to_datetime(
        prepared["timestamp"],
        unit="ms",
        utc=True,
    )

    prepared["minute_start"] = (
        prepared["event_time"]
        .dt.floor(
            f"{instrument_config.bar_minutes}min"
        )
    )

    # Convert feed prices into exact integer price increments.
    prepared["ask_price_ticks"] = np.rint(
        prepared["askPrice"]
        / instrument_config.price_increment
    ).astype("int64")

    prepared["bid_price_ticks"] = np.rint(
        prepared["bidPrice"]
        / instrument_config.price_increment
    ).astype("int64")

    # Twice the midpoint expressed in integer price ticks.
    # Keeping this integer prevents false directional changes.
    prepared["mid_half_ticks"] = (
        prepared["ask_price_ticks"]
        + prepared["bid_price_ticks"]
    )

    prepared["mid"] = (
        prepared["mid_half_ticks"]
        * instrument_config.price_increment
        / 2
    )

    prepared["spread_ticks"] = (
        prepared["ask_price_ticks"]
        - prepared["bid_price_ticks"]
    )

    prepared["spread_units"] = (
        prepared["spread_ticks"]
        * instrument_config.price_increment
        / instrument_config.reporting_unit
    )

    midpoint_values = (
        prepared["mid_half_ticks"]
        .to_numpy()
    )

    if previous_mid_half_ticks is None:
        previous_values = np.empty_like(
            midpoint_values
        )

        previous_values[0] = midpoint_values[0]
        previous_values[1:] = midpoint_values[:-1]

    else:
        previous_values = np.empty_like(
            midpoint_values
        )

        previous_values[0] = (
            previous_mid_half_ticks
        )
        previous_values[1:] = midpoint_values[:-1]

    prepared["tick_direction"] = np.sign(
        midpoint_values - previous_values
    ).astype("int8")

    prepared["is_uptick"] = (
        prepared["tick_direction"] == 1
    )

    prepared["is_downtick"] = (
        prepared["tick_direction"] == -1
    )

    prepared["is_unchanged"] = (
        prepared["tick_direction"] == 0
    )

    prepared["is_locked_quote"] = (
        prepared["ask_price_ticks"]
        == prepared["bid_price_ticks"]
    )

    return prepared

In [8]:
# Cell 8 — Test generic tick preparation on a small sample.
#
# This reads only 100,000 ticks and does not write anything.

SAMPLE_ROW_COUNT = 100_000

tick_sample = pd.read_csv(
    baseline_file,
    nrows=SAMPLE_ROW_COUNT,
    dtype=TICK_DTYPES,
)

prepared_sample = prepare_tick_rows(
    tick_sample,
    instrument,
)


sample_checks = {
    "Rows preserved": (
        len(prepared_sample)
        == len(tick_sample)
    ),
    "Chronological timestamps": (
        prepared_sample[
            "event_time"
        ].is_monotonic_increasing
    ),
    "Ask at or above bid": (
        prepared_sample[
            "ask_price_ticks"
        ]
        >= prepared_sample[
            "bid_price_ticks"
        ]
    ).all(),
    "Non-negative spreads": (
        prepared_sample[
            "spread_ticks"
        ] >= 0
    ).all(),
    "Valid directions": (
        prepared_sample[
            "tick_direction"
        ].isin([-1, 0, 1]).all()
    ),
}


for check_name, passed in sample_checks.items():
    print(
        f"{check_name:<28}",
        "PASS" if passed else "FAIL",
    )


print()
print(
    "Generic tick preparation:",
    (
        "PASS"
        if all(sample_checks.values())
        else "FAIL"
    ),
)

display(
    prepared_sample[
        [
            "event_time",
            "minute_start",
            "askPrice",
            "bidPrice",
            "mid",
            "spread_ticks",
            "spread_units",
            "tick_direction",
        ]
    ].head(10)
)

Rows preserved               PASS
Chronological timestamps     PASS
Ask at or above bid          PASS
Non-negative spreads         PASS
Valid directions             PASS

Generic tick preparation: PASS


,event_time,minute_start,askPrice,bidPrice,mid,spread_ticks,spread_units,tick_direction
0,2010-01-01 00:00:03.964000+00:00,2010-01-01 00:00:00+00:00,1.43293,1.43283,1.432880,10,1.0,0
1,2010-01-01 00:00:05.996000+00:00,2010-01-01 00:00:00+00:00,1.43290,1.43278,1.432840,12,1.2,-1
2,2010-01-01 00:00:10.385000+00:00,2010-01-01 00:00:00+00:00,1.43287,1.43274,1.432805,13,1.3,-1
3,2010-01-01 00:00:15.255000+00:00,2010-01-01 00:00:00+00:00,1.43285,1.43273,1.432790,12,1.2,-1
4,2010-01-01 00:00:19.201000+00:00,2010-01-01 00:00:00+00:00,1.43285,1.43273,1.432790,12,1.2,0
5,2010-01-01 00:00:19.544000+00:00,2010-01-01 00:00:00+00:00,1.43283,1.43273,1.432780,10,1.0,-1
6,2010-01-01 00:00:29.027000+00:00,2010-01-01 00:00:00+00:00,1.43285,1.43273,1.432790,12,1.2,1
7,2010-01-01 00:00:34.424000+00:00,2010-01-01 00:00:00+00:00,1.43284,1.43274,1.432790,10,1.0,0
8,2010-01-01 00:00:37.325000+00:00,2010-01-01 00:00:00+00:00,1.43284,1.43274,1.432790,10,1.0,0
9,2010-01-01 00:00:38.758000+00:00,2010-01-01 00:00:00+00:00,1.43284,1.43272,1.432780,12,1.2,-1


In [9]:
# Cell 9 — Define generic time-bar aggregation.
#
# Column names use "spread_units" rather than "spread_pips".
# The meaning of one unit comes from InstrumentConfig:
#   EUR/USD → one unit is one pip
#   another asset → configurable reporting unit

def aggregate_time_bars(
    prepared_ticks,
    instrument_config,
):
    """Aggregate prepared ticks into configured time bars."""

    if prepared_ticks.empty:
        return pd.DataFrame()

    time_bars = (
        prepared_ticks
        .groupby(
            "minute_start",
            sort=True,
            observed=True,
        )
        .agg(
            mid_open=("mid", "first"),
            mid_high=("mid", "max"),
            mid_low=("mid", "min"),
            mid_close=("mid", "last"),

            bid_open=("bidPrice", "first"),
            bid_high=("bidPrice", "max"),
            bid_low=("bidPrice", "min"),
            bid_close=("bidPrice", "last"),

            ask_open=("askPrice", "first"),
            ask_high=("askPrice", "max"),
            ask_low=("askPrice", "min"),
            ask_close=("askPrice", "last"),

            spread_open_units=(
                "spread_units",
                "first",
            ),
            spread_mean_units=(
                "spread_units",
                "mean",
            ),
            spread_median_units=(
                "spread_units",
                "median",
            ),
            spread_max_units=(
                "spread_units",
                "max",
            ),
            spread_close_units=(
                "spread_units",
                "last",
            ),

            tick_count=("timestamp", "size"),
            uptick_count=("is_uptick", "sum"),
            downtick_count=("is_downtick", "sum"),
            unchanged_count=(
                "is_unchanged",
                "sum",
            ),
            locked_quote_count=(
                "is_locked_quote",
                "sum",
            ),

            ask_volume_mean=("askVolume", "mean"),
            bid_volume_mean=("bidVolume", "mean"),
            ask_volume_close=("askVolume", "last"),
            bid_volume_close=("bidVolume", "last"),
        )
        .reset_index()
    )

    bar_duration = pd.Timedelta(
        minutes=instrument_config.bar_minutes
    )

    time_bars["available_at"] = (
        time_bars["minute_start"]
        + bar_duration
    )

    directional_ticks = (
        time_bars["uptick_count"]
        + time_bars["downtick_count"]
    )

    time_bars["tick_imbalance"] = np.where(
        directional_ticks > 0,
        (
            time_bars["uptick_count"]
            - time_bars["downtick_count"]
        ) / directional_ticks,
        0.0,
    )

    time_bars["ticks_per_second"] = (
        time_bars["tick_count"]
        / bar_duration.total_seconds()
    )

    return time_bars

In [10]:
# Cell 10 — Validate aggregation on the 100,000-tick sample.
#
# The final sample bar may be incomplete because the sample ends
# partway through a minute. That is acceptable for this isolated
# function test; the production chunk processor will carry it.

sample_time_bars = aggregate_time_bars(
    prepared_sample,
    instrument,
)


sample_bar_checks = {
    "Tick counts reconcile": (
        int(sample_time_bars["tick_count"].sum())
        == len(prepared_sample)
    ),
    "Unique bar timestamps": (
        not sample_time_bars[
            "minute_start"
        ].duplicated().any()
    ),
    "Chronological bars": (
        sample_time_bars[
            "minute_start"
        ].is_monotonic_increasing
    ),
    "Availability timing": (
        sample_time_bars["available_at"]
        == (
            sample_time_bars["minute_start"]
            + pd.Timedelta(
                minutes=instrument.bar_minutes
            )
        )
    ).all(),
    "Valid midpoint OHLC": (
        (
            sample_time_bars["mid_low"]
            <= sample_time_bars["mid_open"]
        )
        & (
            sample_time_bars["mid_open"]
            <= sample_time_bars["mid_high"]
        )
        & (
            sample_time_bars["mid_low"]
            <= sample_time_bars["mid_close"]
        )
        & (
            sample_time_bars["mid_close"]
            <= sample_time_bars["mid_high"]
        )
    ).all(),
    "Valid imbalance range": (
        sample_time_bars[
            "tick_imbalance"
        ].between(-1, 1).all()
    ),
}


for check_name, passed in sample_bar_checks.items():
    print(
        f"{check_name:<26}",
        "PASS" if passed else "FAIL",
    )


print()
print(
    "Time-bar aggregation:",
    (
        "PASS"
        if all(sample_bar_checks.values())
        else "FAIL"
    ),
)

print(
    "Sample ticks:",
    f"{len(prepared_sample):,}",
)
print(
    "Sample bars: ",
    f"{len(sample_time_bars):,}",
)

display(sample_time_bars.head())

Tick counts reconcile      PASS
Unique bar timestamps      PASS
Chronological bars         PASS
Availability timing        PASS
Valid midpoint OHLC        PASS
Valid imbalance range      PASS

Time-bar aggregation: PASS
Sample ticks: 100,000
Sample bars:  5,008


,minute_start,mid_open,mid_high,mid_low,mid_close,bid_open,bid_high,bid_low,bid_close,ask_open,...,downtick_count,unchanged_count,locked_quote_count,ask_volume_mean,bid_volume_mean,ask_volume_close,bid_volume_close,available_at,tick_imbalance,ticks_per_second
0,2010-01-01 00:00:00+00:00,1.43288,1.43288,1.43272,1.43272,1.43283,1.43283,1.43267,1.43267,1.43293,...,9,4,0,3.021429,5.307143,1.5,45.4,2010-01-01 00:01:00+00:00,-0.800000,0.233333
1,2010-01-01 00:01:00+00:00,1.43273,1.43290,1.43273,1.43290,1.43268,1.43285,1.43268,1.43284,1.43278,...,0,1,0,2.930769,1.953846,2.5,1.5,2010-01-01 00:02:00+00:00,1.000000,0.216667
2,2010-01-01 00:02:00+00:00,1.43287,1.43287,1.43274,1.43287,1.43280,1.43280,1.43268,1.43280,1.43294,...,4,1,0,2.560000,3.090000,3.2,1.5,2010-01-01 00:03:00+00:00,0.111111,0.166667
3,2010-01-01 00:03:00+00:00,1.43287,1.43287,1.43274,1.43279,1.43280,1.43280,1.43268,1.43273,1.43294,...,7,3,0,3.206667,2.793333,2.9,1.5,2010-01-01 00:04:00+00:00,-0.166667,0.250000
4,2010-01-01 00:04:00+00:00,1.43288,1.43288,1.43282,1.43282,1.43283,1.43283,1.43277,1.43277,1.43293,...,9,3,0,3.058824,3.164706,3.2,1.5,2010-01-01 00:05:00+00:00,-0.285714,0.283333


In [11]:
# Cell 11 — Define a canonical multi-file tick iterator.
#
# The iterator:
#   - reads each CSV in bounded chunks,
#   - validates ordering inside every chunk,
#   - skips timestamps already emitted by an earlier file,
#   - records exactly how many overlap rows were removed.
#
# It does not assume a particular instrument or date range.

def stream_canonical_tick_chunks(
    source_files,
    chunk_size,
    tick_dtypes,
    statistics,
    initial_last_timestamp=None,
):
    """Yield unique chronological tick chunks across CSV files."""

    statistics.clear()

    statistics.update(
        {
            "files_opened": 0,
            "rows_read": 0,
            "overlap_rows_skipped": 0,
            "unique_rows_yielded": 0,
            "first_timestamp_yielded": None,
            "last_timestamp_yielded": (
                initial_last_timestamp
            ),
        }
    )

    last_timestamp_yielded = (
        initial_last_timestamp
    )

    for source_file in source_files:
        statistics["files_opened"] += 1

        chunk_reader = pd.read_csv(
            source_file,
            dtype=tick_dtypes,
            chunksize=chunk_size,
        )

        for raw_chunk in chunk_reader:
            statistics["rows_read"] += len(
                raw_chunk
            )

            if list(raw_chunk.columns) != RAW_TICK_COLUMNS:
                raise ValueError(
                    "Unexpected columns in "
                    f"{source_file}"
                )

            if not raw_chunk[
                "timestamp"
            ].is_monotonic_increasing:
                raise ValueError(
                    "Out-of-order timestamps in "
                    f"{source_file}"
                )

            if raw_chunk[
                "timestamp"
            ].duplicated().any():
                raise ValueError(
                    "Duplicate timestamps in "
                    f"{source_file}"
                )

            if last_timestamp_yielded is not None:
                already_seen = (
                    raw_chunk["timestamp"]
                    <= last_timestamp_yielded
                )

                skipped_rows = int(
                    already_seen.sum()
                )

                statistics[
                    "overlap_rows_skipped"
                ] += skipped_rows

                if skipped_rows:
                    raw_chunk = raw_chunk.loc[
                        ~already_seen
                    ]

            if raw_chunk.empty:
                continue

            raw_chunk = raw_chunk.reset_index(
                drop=True
            )

            first_timestamp = int(
                raw_chunk["timestamp"].iloc[0]
            )

            final_timestamp = int(
                raw_chunk["timestamp"].iloc[-1]
            )

            if (
                last_timestamp_yielded is not None
                and first_timestamp
                <= last_timestamp_yielded
            ):
                raise ValueError(
                    "Canonical stream did not advance."
                )

            if (
                statistics[
                    "first_timestamp_yielded"
                ]
                is None
            ):
                statistics[
                    "first_timestamp_yielded"
                ] = first_timestamp

            statistics[
                "unique_rows_yielded"
            ] += len(raw_chunk)

            statistics[
                "last_timestamp_yielded"
            ] = final_timestamp

            last_timestamp_yielded = (
                final_timestamp
            )

            yield raw_chunk

In [12]:
# Cell 12 — Test overlap removal without scanning the baseline.
#
# The first continuation file contains:
#   - 16,105 rows already present in the baseline,
#   - the remainder of April 5 after the baseline stopped.
#
# The iterator must skip exactly those verified overlap rows.

overlap_test_statistics = {}

overlap_test_chunks = (
    stream_canonical_tick_chunks(
        source_files=[continuation_file],
        chunk_size=25_000,
        tick_dtypes=TICK_DTYPES,
        statistics=overlap_test_statistics,
        initial_last_timestamp=int(
            baseline_row["last_timestamp"]
        ),
    )
)


overlap_test_rows = 0
overlap_test_first_timestamp = None
overlap_test_last_timestamp = None

for test_chunk in overlap_test_chunks:
    overlap_test_rows += len(test_chunk)

    if overlap_test_first_timestamp is None:
        overlap_test_first_timestamp = int(
            test_chunk["timestamp"].iloc[0]
        )

    overlap_test_last_timestamp = int(
        test_chunk["timestamp"].iloc[-1]
    )


overlap_removal_ok = (
    overlap_test_statistics[
        "overlap_rows_skipped"
    ]
    == overlap_row_count
    and overlap_test_statistics["rows_read"]
    == (
        overlap_test_statistics[
            "overlap_rows_skipped"
        ]
        + overlap_test_rows
    )
    and overlap_test_first_timestamp
    > int(baseline_row["last_timestamp"])
    and overlap_test_last_timestamp
    == int(
        first_continuation_row[
            "last_timestamp"
        ]
    )
)


print(
    "Rows read:             ",
    f"{overlap_test_statistics['rows_read']:,}",
)
print(
    "Overlap rows skipped:  ",
    f"{overlap_test_statistics['overlap_rows_skipped']:,}",
)
print(
    "Unique rows retained:  ",
    f"{overlap_test_rows:,}",
)

print()
print(
    "Overlap-removal test:",
    "PASS" if overlap_removal_ok else "FAIL",
)

Rows read:              95,331
Overlap rows skipped:   16,105
Unique rows retained:   79,226

Overlap-removal test: PASS


In [13]:
# Cell 13 — Carry unfinished bars across chunk and file boundaries.
#
# The final bar in each chunk is held back because more ticks
# belonging to that bar may arrive in the next chunk or file.
#
# At the end of the entire stream, the remaining observed bar
# is emitted. Reaching EOF does not prove that the terminal bar
# covers a complete interval; we will check its coverage later.

def produce_time_bar_batches(
    raw_chunks,
    instrument_config,
):
    """Yield time-bar batches from a chronological tick stream."""

    carry_raw = None
    carry_previous_mid = None

    for raw_chunk in raw_chunks:
        if raw_chunk.empty:
            continue

        raw_chunk = (
            raw_chunk[RAW_TICK_COLUMNS]
            .reset_index(drop=True)
        )

        if carry_raw is None:
            combined_raw = raw_chunk
        else:
            combined_raw = pd.concat(
                [carry_raw, raw_chunk],
                ignore_index=True,
            )

        prepared = prepare_tick_rows(
            combined_raw,
            instrument_config,
            previous_mid_half_ticks=carry_previous_mid,
        )

        # Find where the last, potentially unfinished bar starts.
        final_bar_start = (
            prepared["minute_start"].iloc[-1]
        )

        first_carry_position = int(
            np.flatnonzero(
                prepared["minute_start"]
                .eq(final_bar_start)
                .to_numpy()
            )[0]
        )

        completed_ticks = prepared.iloc[
            :first_carry_position
        ]

        # Save the midpoint immediately before the carried rows.
        # If nothing completed, retain the existing prior midpoint.
        if first_carry_position > 0:
            carry_previous_mid = int(
                prepared["mid_half_ticks"].iloc[
                    first_carry_position - 1
                ]
            )

        carry_raw = (
            combined_raw.iloc[first_carry_position:]
            .copy()
            .reset_index(drop=True)
        )

        if not completed_ticks.empty:
            yield aggregate_time_bars(
                completed_ticks,
                instrument_config,
            )

    # Flush only when all chunks from all source files are exhausted.
    if carry_raw is not None and not carry_raw.empty:
        final_prepared = prepare_tick_rows(
            carry_raw,
            instrument_config,
            previous_mid_half_ticks=carry_previous_mid,
        )

        yield aggregate_time_bars(
            final_prepared,
            instrument_config,
        )

In [14]:
# Cell 14 — Verify that chunk boundaries do not change any bars.
#
# The first three chunks contain just one tick each. This tests
# carrying the same unfinished minute through multiple chunks.
#
# Later splits test ordinary boundaries inside active minutes.
# Every output column must match processing the sample at once.

split_positions = [
    0,
    1,
    2,
    3,
    12_345,
    54_321,
    87_654,
    len(tick_sample),
]

artificial_chunks = (
    tick_sample.iloc[start:stop].copy()
    for start, stop in zip(
        split_positions[:-1],
        split_positions[1:],
    )
)

chunked_sample_bars = pd.concat(
    produce_time_bar_batches(
        artificial_chunks,
        instrument,
    ),
    ignore_index=True,
)

# Raise an error immediately if any column differs.
pd.testing.assert_frame_equal(
    chunked_sample_bars,
    sample_time_bars,
    check_dtype=True,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

assert (
    int(chunked_sample_bars["tick_count"].sum())
    == len(tick_sample)
)

assert not chunked_sample_bars[
    "minute_start"
].duplicated().any()

print("Chunk-carry test: PASS")
print("Input chunks:", len(split_positions) - 1)
print("Ticks reconciled:", f"{len(tick_sample):,}")
print("Bars reproduced:", f"{len(chunked_sample_bars):,}")

Chunk-carry test: PASS
Input chunks: 7
Ticks reconciled: 100,000
Bars reproduced: 5,008


In [15]:
# Cell 15 — Build an independent reference for the real-file test.
#
# These two files contain consecutive, non-overlapping periods.
# Together they are small enough to process in memory for testing.
#
# The reference processes all their ticks at once.

next_continuation_file = (
    PROJECT_ROOT
    / canonical_file_table.iloc[2]["relative_path"]
)

reference_raw_ticks = pd.concat(
    [
        pd.read_csv(
            continuation_file,
            dtype=TICK_DTYPES,
        ),
        pd.read_csv(
            next_continuation_file,
            dtype=TICK_DTYPES,
        ),
    ],
    ignore_index=True,
)

# Confirm that the reference itself is strictly chronological.
assert reference_raw_ticks[
    "timestamp"
].is_monotonic_increasing

assert not reference_raw_ticks[
    "timestamp"
].duplicated().any()

reference_file_bars = aggregate_time_bars(
    prepare_tick_rows(
        reference_raw_ticks,
        instrument,
    ),
    instrument,
)

print(
    "Reference ticks:",
    f"{len(reference_raw_ticks):,}",
)
print(
    "Reference bars:",
    f"{len(reference_file_bars):,}",
)

Reference ticks: 564,982
Reference bars: 8,630


In [16]:
# Cell 16 — Test overlap removal and bar carry together.
#
# Feed three files through the production functions:
#   1. The compact overlap reference.
#   2. The full April 5 continuation file.
#   3. The April 6–13 continuation file.
#
# File 2 repeats all ticks from file 1.
# The reader must skip exactly those verified duplicates.
#
# The resulting bars must match the independent reference,
# including tick direction across chunks and the file boundary.

real_file_test_statistics = {}

real_file_test_chunks = stream_canonical_tick_chunks(
    source_files=[
        overlap_reference_file,
        continuation_file,
        next_continuation_file,
    ],
    chunk_size=25_000,
    tick_dtypes=TICK_DTYPES,
    statistics=real_file_test_statistics,
)

real_file_test_bars = pd.concat(
    produce_time_bar_batches(
        real_file_test_chunks,
        instrument,
    ),
    ignore_index=True,
)

pd.testing.assert_frame_equal(
    real_file_test_bars,
    reference_file_bars,
    check_dtype=True,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

assert (
    real_file_test_statistics["overlap_rows_skipped"]
    == overlap_row_count
)

assert (
    real_file_test_statistics["unique_rows_yielded"]
    == len(reference_raw_ticks)
)

assert (
    real_file_test_statistics["rows_read"]
    == len(reference_raw_ticks) + overlap_row_count
)

assert (
    int(real_file_test_bars["tick_count"].sum())
    == len(reference_raw_ticks)
)

assert not real_file_test_bars[
    "minute_start"
].duplicated().any()

print("Real-file pipeline test: PASS")
print(
    "Files tested:",
    real_file_test_statistics["files_opened"],
)
print(
    "Overlap rows removed:",
    f"{real_file_test_statistics['overlap_rows_skipped']:,}",
)
print(
    "Unique ticks reconciled:",
    f"{real_file_test_statistics['unique_rows_yielded']:,}",
)
print(
    "Bars reproduced:",
    f"{len(real_file_test_bars):,}",
)

Real-file pipeline test: PASS
Files tested: 3
Overlap rows removed: 16,105
Unique ticks reconciled: 564,982
Bars reproduced: 8,630


In [17]:
# Cell 17 — Prepare the full production build.
#
# This cell checks paths, compression, disk space, and schema.
# It does not write any data.
#
# Expected row counts come from the completed full-file audit.
# They are reconciliation checks for this dataset, rather than
# instrument settings used by the processing functions.

import shutil


CSV_CHUNK_SIZE = 1_000_000

EXPECTED_UNIQUE_TICK_ROWS = 410_091_208
EXPECTED_OVERLAP_ROWS = overlap_row_count


# Derive output naming from the configured instrument and
# actual observed coverage rather than the raw filenames.
coverage_start = (
    canonical_file_table["first_tick"].min()
)

coverage_end = (
    canonical_file_table["last_tick"].max()
)

output_name = (
    f"{instrument.symbol.lower()}_"
    f"{instrument.bar_minutes}min_"
    f"{coverage_start:%Y-%m-%d}_"
    f"{coverage_end:%Y-%m-%d}.parquet"
)

CANONICAL_OUTPUT_FILE = (
    INTERIM_DATA_DIR / output_name
)

TEMP_CANONICAL_OUTPUT_FILE = (
    CANONICAL_OUTPUT_FILE.with_name(
        f"{CANONICAL_OUTPUT_FILE.stem}"
        ".partial.parquet"
    )
)


INTERIM_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for output_path in [
    CANONICAL_OUTPUT_FILE,
    TEMP_CANONICAL_OUTPUT_FILE,
]:
    if output_path.exists():
        raise FileExistsError(
            f"Output already exists: {output_path}"
        )

if not pa.Codec.is_available("zstd"):
    raise RuntimeError(
        "Zstandard compression is unavailable."
    )


# Lock the schema produced by the tested generic functions.
CANONICAL_SCHEMA = pa.Table.from_pandas(
    sample_time_bars,
    preserve_index=False,
).schema


# Record source identities so the production cell can detect
# files that change during the build.
source_snapshot = {
    source_file: (
        source_file.stat().st_size,
        source_file.stat().st_mtime_ns,
    )
    for source_file in canonical_files
}

source_size_gb = (
    sum(
        size_bytes
        for size_bytes, modified_ns
        in source_snapshot.values()
    )
    / (1024 ** 3)
)

free_space_gb = (
    shutil.disk_usage(INTERIM_DATA_DIR).free
    / (1024 ** 3)
)


print("Production preflight: PASS")
print("Source files:", len(canonical_files))
print("Source size:", f"{source_size_gb:.2f} GiB")
print("Free disk space:", f"{free_space_gb:.2f} GiB")
print("Chunk size:", f"{CSV_CHUNK_SIZE:,}")
print(
    "Expected unique ticks:",
    f"{EXPECTED_UNIQUE_TICK_ROWS:,}",
)
print(
    "Expected overlap removal:",
    f"{EXPECTED_OVERLAP_ROWS:,}",
)
print("Output columns:", len(CANONICAL_SCHEMA))
print("Compression: Zstandard")

print()
print("Temporary output:", TEMP_CANONICAL_OUTPUT_FILE)
print("Final output:", CANONICAL_OUTPUT_FILE)

Production preflight: PASS
Source files: 14
Source size: 14.65 GiB
Free disk space: 783.02 GiB
Chunk size: 1,000,000
Expected unique ticks: 410,091,208
Expected overlap removal: 16,105
Output columns: 30
Compression: Zstandard

Temporary output: C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010-01-01_2026-10-06.partial.parquet
Final output: C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010-01-01_2026-10-06.parquet


In [18]:
# Cell 18 — Build the complete canonical one-minute dataset.
#
# The tested reader removes the verified overlap.
# The tested bar processor carries unfinished bars across chunks
# and files, preserving tick direction.
#
# Progress is printed every ten output batches.
# A failure or interruption retains the partial file for inspection.

import time


build_statistics = {}

ticks_written = 0
bars_written = 0
batches_written = 0

first_bar_written = None
last_bar_written = None

writer = None
output_handle = None

build_start_time = time.perf_counter()


try:
    # Recheck immediately before starting.
    if CANONICAL_OUTPUT_FILE.exists():
        raise FileExistsError(
            f"Final output exists: {CANONICAL_OUTPUT_FILE}"
        )

    for source_file, expected_identity in source_snapshot.items():
        current_stat = source_file.stat()

        current_identity = (
            current_stat.st_size,
            current_stat.st_mtime_ns,
        )

        if current_identity != expected_identity:
            raise RuntimeError(
                f"Source changed after preflight: {source_file}"
            )

    # Exclusive creation prevents overwriting an existing partial.
    output_handle = TEMP_CANONICAL_OUTPUT_FILE.open("xb")

    writer = pq.ParquetWriter(
        output_handle,
        CANONICAL_SCHEMA,
        compression="zstd",
    )

    raw_chunks = stream_canonical_tick_chunks(
        source_files=canonical_files,
        chunk_size=CSV_CHUNK_SIZE,
        tick_dtypes=TICK_DTYPES,
        statistics=build_statistics,
    )

    bar_batches = produce_time_bar_batches(
        raw_chunks,
        instrument,
    )

    for bar_batch in bar_batches:
        timestamps = bar_batch["minute_start"]

        if (
            not timestamps.is_monotonic_increasing
            or timestamps.duplicated().any()
        ):
            raise ValueError(
                "Bar timestamps are unordered or duplicated."
            )

        if (
            last_bar_written is not None
            and timestamps.iloc[0] <= last_bar_written
        ):
            raise ValueError(
                "Output batches overlap or are out of order."
            )

        expected_availability = (
            timestamps
            + pd.Timedelta(minutes=instrument.bar_minutes)
        )

        if not bar_batch["available_at"].equals(
            expected_availability
        ):
            raise ValueError(
                "Invalid bar availability timing."
            )

        direction_counts = (
            bar_batch["uptick_count"]
            + bar_batch["downtick_count"]
            + bar_batch["unchanged_count"]
        )

        if not direction_counts.eq(
            bar_batch["tick_count"]
        ).all():
            raise ValueError(
                "Tick-direction counts do not reconcile."
            )

        arrow_batch = pa.Table.from_pandas(
            bar_batch,
            schema=CANONICAL_SCHEMA,
            preserve_index=False,
            safe=True,
        )

        writer.write_table(arrow_batch)

        ticks_written += int(
            bar_batch["tick_count"].sum()
        )

        bars_written += len(bar_batch)
        batches_written += 1

        if first_bar_written is None:
            first_bar_written = timestamps.iloc[0]

        last_bar_written = timestamps.iloc[-1]

        if batches_written % 10 == 0:
            elapsed = time.perf_counter() - build_start_time

            print(
                f"Batch {batches_written:,} | "
                f"ticks {ticks_written:,} "
                f"({ticks_written / EXPECTED_UNIQUE_TICK_ROWS:.1%}) | "
                f"bars {bars_written:,} | "
                f"elapsed {elapsed / 60:.2f} min",
                flush=True,
            )

    # Close the writer and file before inspecting the Parquet footer.
    writer.close()
    writer = None

    output_handle.close()
    output_handle = None

    if ticks_written != EXPECTED_UNIQUE_TICK_ROWS:
        raise ValueError(
            f"Tick reconciliation failed: {ticks_written:,} "
            f"versus expected {EXPECTED_UNIQUE_TICK_ROWS:,}."
        )

    if (
        build_statistics["unique_rows_yielded"]
        != ticks_written
    ):
        raise ValueError(
            "Reader and writer tick counts differ."
        )

    if (
        build_statistics["overlap_rows_skipped"]
        != EXPECTED_OVERLAP_ROWS
    ):
        raise ValueError(
            "Unexpected overlap-removal count."
        )

    if (
        build_statistics["rows_read"]
        != ticks_written + EXPECTED_OVERLAP_ROWS
    ):
        raise ValueError(
            "Raw row accounting failed."
        )

    if (
        build_statistics["files_opened"]
        != len(canonical_files)
    ):
        raise ValueError(
            "Not all canonical files were processed."
        )

    expected_first_timestamp = int(
        canonical_file_table["first_timestamp"].min()
    )

    expected_last_timestamp = int(
        canonical_file_table["last_timestamp"].max()
    )

    if (
        build_statistics["first_timestamp_yielded"]
        != expected_first_timestamp
        or build_statistics["last_timestamp_yielded"]
        != expected_last_timestamp
    ):
        raise ValueError(
            "Raw stream coverage differs from preflight."
        )

    # Confirm that no source changed while being read.
    for source_file, expected_identity in source_snapshot.items():
        current_stat = source_file.stat()

        if (
            current_stat.st_size,
            current_stat.st_mtime_ns,
        ) != expected_identity:
            raise RuntimeError(
                f"Source changed during the build: {source_file}"
            )

    with pq.ParquetFile(
        TEMP_CANONICAL_OUTPUT_FILE
    ) as completed_file:
        if completed_file.metadata.num_rows != bars_written:
            raise ValueError(
                "Saved Parquet row count does not reconcile."
            )

        if not completed_file.schema_arrow.equals(
            CANONICAL_SCHEMA,
            check_metadata=False,
        ):
            raise ValueError(
                "Saved Parquet schema differs from preflight."
            )

    if CANONICAL_OUTPUT_FILE.exists():
        raise FileExistsError(
            f"Final output now exists: {CANONICAL_OUTPUT_FILE}"
        )

    TEMP_CANONICAL_OUTPUT_FILE.rename(
        CANONICAL_OUTPUT_FILE
    )

except BaseException:
    # Also handle a notebook interruption.
    # Attempt to close the partial file without hiding the
    # original error if closing itself encounters a problem.
    try:
        if writer is not None:
            writer.close()
    except Exception as closing_error:
        print("Writer close warning:", closing_error)
    finally:
        if output_handle is not None:
            output_handle.close()

    print("\nProduction build: STOPPED")

    if TEMP_CANONICAL_OUTPUT_FILE.exists():
        print(
            "Partial output retained:",
            TEMP_CANONICAL_OUTPUT_FILE,
        )

    raise


elapsed_seconds = (
    time.perf_counter() - build_start_time
)

output_size_mb = (
    CANONICAL_OUTPUT_FILE.stat().st_size
    / (1024 ** 2)
)

print()
print("Production build: PASS")
print("Source files:", build_statistics["files_opened"])
print("Raw rows read:", f"{build_statistics['rows_read']:,}")
print(
    "Overlap rows removed:",
    f"{build_statistics['overlap_rows_skipped']:,}",
)
print("Unique ticks reconciled:", f"{ticks_written:,}")
print("Bars written:", f"{bars_written:,}")
print("First bar:", first_bar_written)
print("Last bar:", last_bar_written)
print("Output size:", f"{output_size_mb:,.2f} MiB")
print("Elapsed time:", f"{elapsed_seconds / 60:.2f} minutes")
print("Final output:", CANONICAL_OUTPUT_FILE)

Batch 10 | ticks 9,999,993 (2.4%) | bars 400,592 | elapsed 0.16 min
Batch 20 | ticks 19,999,996 (4.9%) | bars 558,908 | elapsed 0.26 min
Batch 30 | ticks 29,999,969 (7.3%) | bars 680,772 | elapsed 0.36 min
Batch 40 | ticks 39,999,976 (9.8%) | bars 813,029 | elapsed 0.46 min
Batch 50 | ticks 49,999,934 (12.2%) | bars 959,954 | elapsed 0.56 min
Batch 60 | ticks 59,999,957 (14.6%) | bars 1,160,044 | elapsed 0.66 min
Batch 70 | ticks 69,999,961 (17.1%) | bars 1,349,299 | elapsed 0.76 min
Batch 80 | ticks 79,999,984 (19.5%) | bars 1,595,350 | elapsed 0.86 min
Batch 90 | ticks 89,999,959 (21.9%) | bars 1,795,294 | elapsed 0.97 min
Batch 100 | ticks 99,999,873 (24.4%) | bars 1,953,487 | elapsed 1.07 min
Batch 110 | ticks 109,999,937 (26.8%) | bars 2,103,133 | elapsed 1.17 min
Batch 120 | ticks 119,999,848 (29.3%) | bars 2,248,971 | elapsed 1.26 min
Batch 130 | ticks 128,056,670 (31.2%) | bars 2,365,851 | elapsed 1.35 min
Batch 140 | ticks 138,056,703 (33.7%) | bars 2,434,535 | elapsed 1.46 mi

In [19]:
# Cell 19 — Validate core accounting and timestamps from disk.
#
# Read the completed Parquet rather than relying on the
# production loop's in-memory counters.
#
# Timestamp gaps are allowed here. We will inventory them next;
# these checks do not imply uninterrupted market coverage.

with pq.ParquetFile(
    CANONICAL_OUTPUT_FILE
) as saved_parquet:
    saved_metadata_rows = saved_parquet.metadata.num_rows
    saved_metadata_columns = saved_parquet.metadata.num_columns

    saved_schema_ok = saved_parquet.schema_arrow.equals(
        CANONICAL_SCHEMA,
        check_metadata=False,
    )


canonical_core = pd.read_parquet(
    CANONICAL_OUTPUT_FILE,
    columns=[
        "minute_start",
        "available_at",
        "tick_count",
    ],
)

bar_duration = pd.Timedelta(
    minutes=instrument.bar_minutes
)

expected_first_bar = coverage_start.floor(
    f"{instrument.bar_minutes}min"
)

expected_last_bar = coverage_end.floor(
    f"{instrument.bar_minutes}min"
)

saved_tick_total = int(
    canonical_core["tick_count"].sum()
)


core_checks = {
    "Schema matches": saved_schema_ok,
    "Column count": (
        saved_metadata_columns == len(CANONICAL_SCHEMA)
    ),
    "Saved row count": (
        len(canonical_core)
        == saved_metadata_rows
        == bars_written
    ),
    "No missing core values": (
        not canonical_core.isna().any().any()
    ),
    "Chronological bars": (
        canonical_core[
            "minute_start"
        ].is_monotonic_increasing
    ),
    "Unique timestamps": (
        not canonical_core[
            "minute_start"
        ].duplicated().any()
    ),
    "Aligned bar timestamps": (
        canonical_core["minute_start"]
        == canonical_core["minute_start"].dt.floor(
            f"{instrument.bar_minutes}min"
        )
    ).all(),
    "Availability timing": (
        canonical_core["available_at"]
        == canonical_core["minute_start"] + bar_duration
    ).all(),
    "Positive tick counts": (
        canonical_core["tick_count"] > 0
    ).all(),
    "Tick reconciliation": (
        saved_tick_total == EXPECTED_UNIQUE_TICK_ROWS
    ),
    "Coverage boundaries": (
        canonical_core["minute_start"].iloc[0]
        == expected_first_bar
        and canonical_core["minute_start"].iloc[-1]
        == expected_last_bar
    ),
    "Partial file absent": (
        not TEMP_CANONICAL_OUTPUT_FILE.exists()
    ),
}


for check_name, passed in core_checks.items():
    print(
        f"{check_name:<28}",
        "PASS" if passed else "FAIL",
    )

# Stop subsequent work if any independent check fails.
assert all(core_checks.values()), (
    "Saved-file validation failed. Inspect the checks above."
)

print()
print("Core file validation: PASS")
print("Saved bars:", f"{len(canonical_core):,}")
print("Saved tick total:", f"{saved_tick_total:,}")
print("First bar:", canonical_core["minute_start"].iloc[0])
print("Last bar:", canonical_core["minute_start"].iloc[-1])

Schema matches               PASS
Column count                 PASS
Saved row count              PASS
No missing core values       PASS
Chronological bars           PASS
Unique timestamps            PASS
Aligned bar timestamps       PASS
Availability timing          PASS
Positive tick counts         PASS
Tick reconciliation          PASS
Coverage boundaries          PASS
Partial file absent          PASS

Core file validation: PASS
Saved bars: 6,206,712
Saved tick total: 410,091,208
First bar: 2010-01-01 00:00:00+00:00
Last bar: 2026-10-06 23:59:00+00:00


In [20]:
# Cell 20 — Define checks for one batch of saved bars.
#
# Each result counts rows that violate a relationship.
# All violation counts should be zero.

def count_bar_violations(bars, instrument_config):
    """Return row-level violation counts for saved bar values."""

    violations = {}

    numeric_values = bars.to_numpy(dtype="float64")

    violations["Non-finite numeric values"] = int(
        (~np.isfinite(numeric_values).all(axis=1)).sum()
    )

    # Open and close must lie inside each price range.
    for price_type in ["mid", "bid", "ask"]:
        price_columns = [
            f"{price_type}_open",
            f"{price_type}_high",
            f"{price_type}_low",
            f"{price_type}_close",
        ]

        valid_prices = (
            bars[price_columns].gt(0).all(axis=1)
            & bars[f"{price_type}_low"].le(
                bars[f"{price_type}_open"]
            )
            & bars[f"{price_type}_low"].le(
                bars[f"{price_type}_close"]
            )
            & bars[f"{price_type}_high"].ge(
                bars[f"{price_type}_open"]
            )
            & bars[f"{price_type}_high"].ge(
                bars[f"{price_type}_close"]
            )
        )

        violations[f"Invalid {price_type} OHLC"] = int(
            (~valid_prices).sum()
        )

    count_columns = [
        "tick_count",
        "uptick_count",
        "downtick_count",
        "unchanged_count",
        "locked_quote_count",
    ]

    counts = bars[count_columns]

    valid_counts = (
        counts.ge(0).all(axis=1)
        & counts.mod(1).eq(0).all(axis=1)
        & bars["tick_count"].gt(0)
        & bars["locked_quote_count"].le(bars["tick_count"])
        & (
            bars["uptick_count"]
            + bars["downtick_count"]
            + bars["unchanged_count"]
        ).eq(bars["tick_count"])
    )

    violations["Invalid tick counts"] = int(
        (~valid_counts).sum()
    )

    spread_columns = [
        "spread_open_units",
        "spread_mean_units",
        "spread_median_units",
        "spread_max_units",
        "spread_close_units",
    ]

    # Allow negligible floating-point rounding in spread statistics.
    spread_tolerance = (
        instrument_config.price_increment
        / instrument_config.reporting_unit
        * 1e-8
    )

    spreads = bars[spread_columns]

    valid_spreads = (
        spreads.ge(0).all(axis=1)
        & spreads.le(
            bars["spread_max_units"] + spread_tolerance,
            axis=0,
        ).all(axis=1)
    )

    violations["Invalid spread statistics"] = int(
        (~valid_spreads).sum()
    )

    # Check midpoint and spread against executable opening/closing quotes.
    for boundary in ["open", "close"]:
        expected_mid = (
            bars[f"ask_{boundary}"]
            + bars[f"bid_{boundary}"]
        ) / 2

        expected_spread = (
            bars[f"ask_{boundary}"]
            - bars[f"bid_{boundary}"]
        ) / instrument_config.reporting_unit

        violations[f"{boundary} midpoint mismatch"] = int(
            (~np.isclose(
                bars[f"mid_{boundary}"],
                expected_mid,
                rtol=1e-12,
                atol=instrument_config.price_increment * 1e-8,
            )).sum()
        )

        violations[f"{boundary} spread mismatch"] = int(
            (~np.isclose(
                bars[f"spread_{boundary}_units"],
                expected_spread,
                rtol=1e-10,
                atol=spread_tolerance,
            )).sum()
        )

    directional_count = (
        bars["uptick_count"] + bars["downtick_count"]
    )

    expected_imbalance = (
        (bars["uptick_count"] - bars["downtick_count"])
        / directional_count.replace(0, np.nan)
    ).fillna(0.0)

    violations["Tick imbalance mismatch"] = int(
        (~np.isclose(
            bars["tick_imbalance"],
            expected_imbalance,
            rtol=1e-12,
            atol=1e-12,
        )).sum()
    )

    expected_tick_rate = (
        bars["tick_count"]
        / (instrument_config.bar_minutes * 60)
    )

    violations["Tick rate mismatch"] = int(
        (~np.isclose(
            bars["ticks_per_second"],
            expected_tick_rate,
            rtol=1e-12,
            atol=1e-12,
        )).sum()
    )

    volume_columns = [
        "ask_volume_mean",
        "bid_volume_mean",
        "ask_volume_close",
        "bid_volume_close",
    ]

    violations["Negative quote volumes"] = int(
        (~bars[volume_columns].ge(0).all(axis=1)).sum()
    )

    return violations

In [21]:
# Cell 21 — Scan all saved bars using bounded memory.
#
# Read numeric columns in batches of 100,000 bars.
# The checks validate every row without loading all 30 columns
# for the entire dataset at once.

numeric_columns = [
    column
    for column in CANONICAL_SCHEMA.names
    if column not in ["minute_start", "available_at"]
]

bar_violation_totals = {}
validated_bar_count = 0
saved_locked_quote_total = 0
saved_max_spread_units = 0.0

with pq.ParquetFile(CANONICAL_OUTPUT_FILE) as saved_parquet:
    for batch_number, arrow_batch in enumerate(
        saved_parquet.iter_batches(
            batch_size=100_000,
            columns=numeric_columns,
        ),
        start=1,
    ):
        saved_bars = arrow_batch.to_pandas()

        batch_violations = count_bar_violations(
            saved_bars,
            instrument,
        )

        for check_name, count in batch_violations.items():
            bar_violation_totals[check_name] = (
                bar_violation_totals.get(check_name, 0)
                + count
            )

        validated_bar_count += len(saved_bars)

        saved_locked_quote_total += int(
            saved_bars["locked_quote_count"].sum()
        )

        saved_max_spread_units = max(
            saved_max_spread_units,
            float(saved_bars["spread_max_units"].max()),
        )

        if batch_number % 20 == 0:
            print(
                f"Validated {validated_bar_count:,} bars...",
                flush=True,
            )


print()

for check_name, violation_count in bar_violation_totals.items():
    print(f"{check_name:<30} {violation_count:,}")

assert validated_bar_count == saved_metadata_rows
assert all(
    count == 0
    for count in bar_violation_totals.values()
), "Saved-bar validation found violations."

print()
print("Full saved-bar validation: PASS")
print("Bars checked:", f"{validated_bar_count:,}")
print("Locked quotes retained:", f"{saved_locked_quote_total:,}")
print(
    "Maximum spread:",
    f"{saved_max_spread_units:.2f}",
    instrument.reporting_unit_name,
)

Validated 2,000,000 bars...
Validated 4,000,000 bars...
Validated 6,000,000 bars...

Non-finite numeric values      0
Invalid mid OHLC               0
Invalid bid OHLC               0
Invalid ask OHLC               0
Invalid tick counts            0
Invalid spread statistics      0
open midpoint mismatch         0
open spread mismatch           0
close midpoint mismatch        0
close spread mismatch          0
Tick imbalance mismatch        0
Tick rate mismatch             0
Negative quote volumes         0

Full saved-bar validation: PASS
Bars checked: 6,206,712
Locked quotes retained: 2,359
Maximum spread: 34.40 pip


In [22]:
# Cell 22 — Build the full-history gap inventory.
#
# Compare consecutive observed bar timestamps.
# A gap contains the scheduled bar slots between those rows.
#
# This does not insert or fill any missing bars.

bar_spacing = canonical_core["minute_start"].diff()

gap_positions = np.flatnonzero(
    bar_spacing.gt(bar_duration).to_numpy()
)

previous_observed = (
    canonical_core["minute_start"]
    .iloc[gap_positions - 1]
    .reset_index(drop=True)
)

next_observed = (
    canonical_core["minute_start"]
    .iloc[gap_positions]
    .reset_index(drop=True)
)

canonical_gap_table = pd.DataFrame(
    {
        "previous_observed": previous_observed,
        "next_observed": next_observed,
    }
)

canonical_gap_table["first_missing"] = (
    canonical_gap_table["previous_observed"]
    + bar_duration
)

canonical_gap_table["last_missing"] = (
    canonical_gap_table["next_observed"]
    - bar_duration
)

canonical_gap_table["missing_bars"] = (
    (
        canonical_gap_table["next_observed"]
        - canonical_gap_table["previous_observed"]
    )
    // bar_duration
    - 1
).astype("int64")

canonical_gap_table["missing_minutes"] = (
    canonical_gap_table["missing_bars"]
    * instrument.bar_minutes
)


# Independently reconcile all absent slots between the first
# and last observed bar, including weekends and holidays.
calendar_bar_slots = int(
    (
        canonical_core["minute_start"].iloc[-1]
        - canonical_core["minute_start"].iloc[0]
    )
    // bar_duration
) + 1

total_missing_bars = int(
    canonical_gap_table["missing_bars"].sum()
)

assert (
    len(canonical_core) + total_missing_bars
    == calendar_bar_slots
), "Gap accounting does not reconcile."


# Summarize gap duration without assigning a cause.
canonical_gap_table["duration_group"] = pd.cut(
    canonical_gap_table["missing_minutes"],
    bins=[
        0, 1, 5, 15, 60, 180, 720, 1_440, np.inf,
    ],
    labels=[
        "1 minute",
        "2–5 minutes",
        "6–15 minutes",
        "16–60 minutes",
        "61–180 minutes",
        "181–720 minutes",
        "721–1,440 minutes",
        "Over 1,440 minutes",
    ],
)

canonical_gap_summary = (
    canonical_gap_table
    .groupby("duration_group", observed=False)
    .agg(
        gap_count=("missing_bars", "size"),
        missing_minutes=("missing_minutes", "sum"),
    )
    .reset_index()
)

total_missing_minutes = int(
    canonical_gap_table["missing_minutes"].sum()
)

canonical_gap_summary["share_of_missing_time_pct"] = (
    100
    * canonical_gap_summary["missing_minutes"]
    / max(total_missing_minutes, 1)
)


print("Gap accounting: PASS")
print("Observed bars:", f"{len(canonical_core):,}")
print("Calendar bar slots:", f"{calendar_bar_slots:,}")
print("Number of gaps:", f"{len(canonical_gap_table):,}")
print("Missing bar slots:", f"{total_missing_bars:,}")
print("Missing minutes:", f"{total_missing_minutes:,}")

display(canonical_gap_summary)

print("\nLargest 15 gaps:")

display(
    canonical_gap_table
    .sort_values("missing_minutes", ascending=False)
    .head(15)[
        [
            "previous_observed",
            "next_observed",
            "missing_minutes",
        ]
    ]
    .reset_index(drop=True)
)

Gap accounting: PASS
Observed bars: 6,206,712
Calendar bar slots: 8,817,120
Number of gaps: 15,222
Missing bar slots: 2,610,408
Missing minutes: 2,610,408


,duration_group,gap_count,missing_minutes,share_of_missing_time_pct
0,1 minute,11042,11042,0.422999
1,2–5 minutes,2423,6052,0.231841
2,6–15 minutes,123,995,0.038117
3,16–60 minutes,704,41397,1.585844
4,61–180 minutes,37,4307,0.164993
5,181–720 minutes,3,831,0.031834
6,"721–1,440 minutes",12,12489,0.478431
7,"Over 1,440 minutes",878,2533295,97.045941



Largest 15 gaps:


,previous_observed,next_observed,missing_minutes
0,2015-12-31 21:59:00+00:00,2016-01-03 22:00:00+00:00,4320
1,2023-12-29 21:59:00+00:00,2024-01-01 22:00:00+00:00,4320
2,2017-12-29 21:59:00+00:00,2018-01-01 22:00:00+00:00,4320
3,2020-12-31 21:59:00+00:00,2021-01-03 22:00:00+00:00,4320
4,2020-12-25 07:59:00+00:00,2020-12-27 22:06:00+00:00,3726
5,2015-12-25 07:59:00+00:00,2015-12-27 22:02:00+00:00,3722
6,2019-05-24 20:59:00+00:00,2019-05-26 23:59:00+00:00,3059
7,2010-02-26 21:59:00+00:00,2010-03-01 00:00:00+00:00,3000
8,2023-12-22 21:58:00+00:00,2023-12-24 23:00:00+00:00,2941
9,2013-11-15 21:59:00+00:00,2013-11-17 23:00:00+00:00,2940


In [23]:
# Cell 23 — Identify isolated, hour-aligned coverage gaps.
#
# Select gaps containing exactly one absent clock hour.
# These are repair candidates; confirming their cause requires
# comparing against Dukascopy's source data.
#
# Longer gaps remain recorded in canonical_gap_table.

one_hour = pd.Timedelta(hours=1)

hour_aligned = (
    canonical_gap_table["first_missing"]
    == canonical_gap_table["first_missing"].dt.floor("h")
)

one_complete_hour = (
    canonical_gap_table["last_missing"] + bar_duration
    == canonical_gap_table["first_missing"] + one_hour
)

missing_hour_candidates = (
    canonical_gap_table.loc[
        hour_aligned & one_complete_hour,
        [
            "previous_observed",
            "next_observed",
            "first_missing",
            "last_missing",
            "missing_bars",
            "missing_minutes",
        ],
    ]
    .copy()
    .sort_values("first_missing")
    .reset_index(drop=True)
)

missing_hour_candidates.insert(
    0,
    "instrument",
    instrument.symbol,
)

missing_hour_candidates["source"] = instrument.source

missing_hour_candidates["status"] = (
    "candidate_missing_source_block"
)

missing_hour_candidates["year"] = (
    missing_hour_candidates["first_missing"].dt.year
)

missing_hour_candidates["weekday"] = (
    missing_hour_candidates["first_missing"].dt.day_name()
)

missing_hour_candidates["hour_utc"] = (
    missing_hour_candidates["first_missing"].dt.hour
)


# Confirm that candidate hours are unique and exactly one hour.
assert not missing_hour_candidates[
    "first_missing"
].duplicated().any()

assert missing_hour_candidates[
    "missing_minutes"
].eq(60).all()


print("Missing-hour candidate extraction: PASS")
print(
    "Isolated missing clock hours:",
    f"{len(missing_hour_candidates):,}",
)

print("\nCandidates by year:")

display(
    missing_hour_candidates
    .groupby("year")
    .size()
    .rename("candidate_hours")
    .reset_index()
)

print("\nFirst 10 candidates:")

display(
    missing_hour_candidates[
        [
            "instrument",
            "first_missing",
            "last_missing",
            "weekday",
        ]
    ].head(10)
)

Missing-hour candidate extraction: PASS
Isolated missing clock hours: 676

Candidates by year:


,year,candidate_hours
0,2010,262
1,2011,26
2,2012,73
3,2013,83
4,2014,106
5,2015,60
6,2016,11
7,2017,2
8,2019,4
9,2023,2



First 10 candidates:


,instrument,first_missing,last_missing,weekday
0,EURUSD,2010-01-01 17:00:00+00:00,2010-01-01 17:59:00+00:00,Friday
1,EURUSD,2010-01-01 20:00:00+00:00,2010-01-01 20:59:00+00:00,Friday
2,EURUSD,2010-01-04 02:00:00+00:00,2010-01-04 02:59:00+00:00,Monday
3,EURUSD,2010-01-04 04:00:00+00:00,2010-01-04 04:59:00+00:00,Monday
4,EURUSD,2010-01-07 02:00:00+00:00,2010-01-07 02:59:00+00:00,Thursday
5,EURUSD,2010-01-08 01:00:00+00:00,2010-01-08 01:59:00+00:00,Friday
6,EURUSD,2010-01-11 02:00:00+00:00,2010-01-11 02:59:00+00:00,Monday
7,EURUSD,2010-01-11 21:00:00+00:00,2010-01-11 21:59:00+00:00,Monday
8,EURUSD,2010-01-14 21:00:00+00:00,2010-01-14 21:59:00+00:00,Thursday
9,EURUSD,2010-01-15 20:00:00+00:00,2010-01-15 20:59:00+00:00,Friday


In [24]:
# Cell 24 — Reconcile the original repair manifest.
#
# Compare actual UTC hour timestamps, rather than counts alone.
# Every original candidate should appear in the rebuilt dataset.

baseline_end = pd.to_datetime(
    int(baseline_row["last_timestamp"]),
    unit="ms",
    utc=True,
)

OLD_HOUR_MANIFEST_FILE = (
    REPORT_TABLES_DIR
    / (
        f"{instrument.symbol.lower()}_"
        f"missing_hour_manifest_"
        f"{coverage_start:%Y}_{baseline_end:%Y}.csv"
    )
)

old_hour_manifest = pd.read_csv(
    OLD_HOUR_MANIFEST_FILE
)

old_hour_starts = pd.DatetimeIndex(
    pd.to_datetime(
        old_hour_manifest["missing_hour_start_utc"],
        utc=True,
    )
)

rebuilt_baseline_hour_starts = pd.DatetimeIndex(
    missing_hour_candidates.loc[
        missing_hour_candidates["first_missing"]
        <= baseline_end,
        "first_missing",
    ]
)

missing_from_rebuild = old_hour_starts.difference(
    rebuilt_baseline_hour_starts
)

additional_baseline_hours = (
    rebuilt_baseline_hour_starts.difference(
        old_hour_starts
    )
)

assert old_hour_starts.is_unique
assert rebuilt_baseline_hour_starts.is_unique

assert len(missing_from_rebuild) == 0, (
    "Original candidate hours are missing from the rebuild."
)

assert len(additional_baseline_hours) == 0, (
    "The baseline candidate-hour list has changed."
)

print("Original manifest reconciliation: PASS")
print("Original candidate hours:", len(old_hour_starts))
print(
    "Rebuilt baseline candidate hours:",
    len(rebuilt_baseline_hour_starts),
)
print(
    "Additional hours after baseline:",
    len(missing_hour_candidates)
    - len(rebuilt_baseline_hour_starts),
)

Original manifest reconciliation: PASS
Original candidate hours: 620
Rebuilt baseline candidate hours: 620
Additional hours after baseline: 56
